In [2]:
!pip install dice-ml

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 27.6 MB/s eta 0:00:00


In [3]:
import pandas as pd
import numpy as np
import joblib

rf_model = joblib.load("random_forest_final_model.pkl")

X_test_scaled = pd.read_csv("X_test_scaled.csv")
rfm_df = pd.read_csv("rfm_segmented_data.csv")

In [4]:
def churn_risk_label(prob):
    if prob >= 0.70:
        return "High Churn Risk"
    elif prob >= 0.40:
        return "Medium Churn Risk"
    else:
        return "Low Churn Risk"

In [5]:
def segment_weight(segment):
    if segment == "High-Value Customers Losing Interest":
        return 25
    elif segment == "Previously Valuable Customers":
        return 20
    elif segment == "Frequent Customers Losing Interest":
        return 15
    elif segment == "Inactive Customers":
        return 10
    elif segment == "High-Value Occasional Customers":
        return 5
    else:
        return 0

In [6]:
def priority_score(prob, segment):
    return round((prob * 100) + segment_weight(segment), 1)

In [7]:
def priority_label(churn_prob, segment):

    if churn_prob >= 0.70:
        return "Immediate Action"

    elif churn_prob >= 0.40:
        if segment in [
            "Previously Valuable Customers",
            "Frequent Customers Losing Interest",
            "High-Value Customers Losing Interest",
            "Inactive Customers"
        ]:
            return "Immediate Action"
        else:
            return "Follow Up Soon"

    else:
        return "Monitor"

In [8]:
def final_manager_action(segment, risk):

    if segment == "High-Value Active Customers":
        if risk == "High Churn Risk":
            return "Investigate churn drivers immediately and provide premium loyalty support."
        elif risk == "Medium Churn Risk":
            return "Send loyalty reward and monitor satisfaction."
        else:
            return "Maintain VIP relationship with regular loyalty benefits."

    elif segment == "Frequent Low-Value Customers":
        if risk == "High Churn Risk":
            return "Use personalised bundle offer and retention discount."
        elif risk == "Medium Churn Risk":
            return "Use cross-sell and upsell campaigns."
        else:
            return "Encourage higher basket value through product bundles."

    elif segment == "High-Value Occasional Customers":
        if risk == "High Churn Risk":
            return "Send premium personalised retention offer."
        elif risk == "Medium Churn Risk":
            return "Encourage more frequent purchases with targeted offers."
        else:
            return "Maintain engagement with personalised product recommendations."

    elif segment == "New and Developing Customers":
        if risk == "High Churn Risk":
            return "Provide onboarding support and first-purchase retention incentive."
        elif risk == "Medium Churn Risk":
            return "Send welcome offer and nurture campaign."
        else:
            return "Continue onboarding and monitor engagement."

    elif segment == "Previously Valuable Customers":
        if risk == "High Churn Risk":
            return "Run priority win-back campaign with personalised discount and direct follow-up."
        elif risk == "Medium Churn Risk":
            return "Send re-engagement campaign with loyalty incentive."
        else:
            return "Monitor behaviour and use light-touch reactivation message."

    elif segment == "Frequent Customers Losing Interest":
        if risk == "High Churn Risk":
            return "Send urgent re-engagement campaign with loyalty reward."
        elif risk == "Medium Churn Risk":
            return "Send reminder campaign and personalised offer."
        else:
            return "Monitor engagement trend before offering discount."

    elif segment == "High-Value Customers Losing Interest":
        if risk == "High Churn Risk":
            return "Use premium retention offer and personal outreach immediately."
        elif risk == "Medium Churn Risk":
            return "Send personalised high-value recovery offer."
        else:
            return "Monitor closely and maintain premium communication."

    elif segment == "Inactive Customers":
        if risk == "High Churn Risk":
            return "Use low-cost win-back campaign or automated reactivation offer."
        elif risk == "Medium Churn Risk":
            return "Send reminder campaign with small incentive."
        else:
            return "No immediate costly action; monitor or include in low-cost automated campaign."

    else:
        return "Monitor customer behaviour and continue regular engagement."

In [9]:
recommendation_results = []

for i in range(len(X_test_scaled)):

    customer_scaled = X_test_scaled.iloc[[i]]
    churn_prob = rf_model.predict_proba(customer_scaled)[0][1]

    customer_original = rfm_df.iloc[i]
    segment = customer_original["CustomerSegment"]

    risk = churn_risk_label(churn_prob)
    score = priority_score(churn_prob, segment)
    priority = priority_label(churn_prob, segment)
    action = final_manager_action(segment, risk)

    recommendation_results.append({
        "Customer_Index": i,
        "Churn_Probability": round(churn_prob, 3),
        "Churn_Risk": risk,
        "Customer_Behaviour_Segment": segment,
        "Priority_Score": score,
        "Priority_Label": priority,
        "Final_Manager_Action": action
    })

final_recommendation_df = pd.DataFrame(recommendation_results)

final_recommendation_df.head()

,Customer_Index,Churn_Probability,Churn_Risk,Customer_Behaviour_Segment,Priority_Score,Priority_Label,Final_Manager_Action
0,0,0.005,Low Churn Risk,Inactive Customers,10.5,Monitor,No immediate costly action; monitor or include...
1,1,0.055,Low Churn Risk,New and Developing Customers,5.5,Monitor,Continue onboarding and monitor engagement.
2,2,0.045,Low Churn Risk,New and Developing Customers,4.5,Monitor,Continue onboarding and monitor engagement.
3,3,0.050,Low Churn Risk,New and Developing Customers,5.0,Monitor,Continue onboarding and monitor engagement.
4,4,0.155,Low Churn Risk,New and Developing Customers,15.5,Monitor,Continue onboarding and monitor engagement.


In [10]:
final_recommendation_df[
    ["Customer_Index", "Churn_Probability", "Churn_Risk",
     "Customer_Behaviour_Segment", "Priority_Score",
     "Priority_Label", "Final_Manager_Action"]
].head(20)

,Customer_Index,Churn_Probability,Churn_Risk,Customer_Behaviour_Segment,Priority_Score,Priority_Label,Final_Manager_Action
0,0,0.005,Low Churn Risk,Inactive Customers,10.5,Monitor,No immediate costly action; monitor or include...
1,1,0.055,Low Churn Risk,New and Developing Customers,5.5,Monitor,Continue onboarding and monitor engagement.
2,2,0.045,Low Churn Risk,New and Developing Customers,4.5,Monitor,Continue onboarding and monitor engagement.
3,3,0.050,Low Churn Risk,New and Developing Customers,5.0,Monitor,Continue onboarding and monitor engagement.
4,4,0.155,Low Churn Risk,New and Developing Customers,15.5,Monitor,Continue onboarding and monitor engagement.
5,5,0.035,Low Churn Risk,Frequent Customers Losing Interest,18.5,Monitor,Monitor engagement trend before offering disco...
6,6,0.145,Low Churn Risk,New and Developing Customers,14.5,Monitor,Continue onboarding and monitor engagement.
7,7,0.040,Low Churn Risk,New and Developing Customers,4.0,Monitor,Continue onboarding and monitor engagement.
8,8,0.130,Low Churn Risk,New and Developing Customers,13.0,Monitor,Continue onboarding and monitor engagement.
9,9,0.035,Low Churn Risk,New and Developing Customers,3.5,Monitor,Continue onboarding and monitor engagement.


In [11]:
final_recommendation_df[
    final_recommendation_df["Churn_Risk"] == "High Churn Risk"
][
    ["Customer_Index", "Churn_Probability", "Churn_Risk",
     "Customer_Behaviour_Segment", "Priority_Label",
     "Final_Manager_Action"]
].head(20)

,Customer_Index,Churn_Probability,Churn_Risk,Customer_Behaviour_Segment,Priority_Label,Final_Manager_Action
11,11,0.785,High Churn Risk,New and Developing Customers,Immediate Action,Provide onboarding support and first-purchase ...
41,41,0.735,High Churn Risk,Inactive Customers,Immediate Action,Use low-cost win-back campaign or automated re...
53,53,0.800,High Churn Risk,Previously Valuable Customers,Immediate Action,Run priority win-back campaign with personalis...
59,59,0.850,High Churn Risk,New and Developing Customers,Immediate Action,Provide onboarding support and first-purchase ...
79,79,0.760,High Churn Risk,High-Value Occasional Customers,Immediate Action,Send premium personalised retention offer.
81,81,0.745,High Churn Risk,New and Developing Customers,Immediate Action,Provide onboarding support and first-purchase ...
85,85,0.915,High Churn Risk,Previously Valuable Customers,Immediate Action,Run priority win-back campaign with personalis...
86,86,0.900,High Churn Risk,Previously Valuable Customers,Immediate Action,Run priority win-back campaign with personalis...
88,88,0.925,High Churn Risk,High-Value Active Customers,Immediate Action,Investigate churn drivers immediately and prov...
92,92,0.795,High Churn Risk,Inactive Customers,Immediate Action,Use low-cost win-back campaign or automated re...


In [12]:
final_recommendation_df.to_csv(
    "final_recommendation_engine_output.csv",
    index=False
)

In [13]:
!pip install dice-ml

In [14]:
import dice_ml
from dice_ml import Dice

In [16]:
# Load files needed for DiCE

rf_model = joblib.load("random_forest_final_model.pkl")

X_train_smote = pd.read_csv("X_train_smote.csv")
y_train_smote = pd.read_csv("y_train_smote.csv").squeeze()

X_test_scaled = pd.read_csv("X_test_scaled.csv")

final_recommendation_df = pd.read_csv("final_recommendation_engine_output.csv")

In [17]:
# Prepare DiCE training dataframe

dice_train_df = X_train_smote.copy()
dice_train_df["Churn"] = y_train_smote

dice_train_df.head()

,Tenure,CityTier,WarehouseToHome,HourSpendOnApp,NumberOfDeviceRegistered,SatisfactionScore,NumberOfAddress,Complain,OrderAmountHikeFromlastYear,CouponUsed,...,M_Score_3,M_Score_4,CustomerSegment_Frequent Low-Value Customers,CustomerSegment_High-Value Active Customers,CustomerSegment_High-Value Customers Losing Interest,CustomerSegment_High-Value Occasional Customers,CustomerSegment_Inactive Customers,CustomerSegment_New and Developing Customers,CustomerSegment_Previously Valuable Customers,Churn
0,-0.135398,1.453559,0.043177,-1.325711,-0.675976,-1.493573,-0.856755,-0.623912,-0.468858,3.386635,...,-0.577350,1.718821,-0.319423,-0.375274,-0.304423,-0.340118,-0.293405,-0.552042,2.058448,0
1,-0.494759,1.453559,-0.318966,-2.742656,-0.675976,0.673881,-1.246653,-0.623912,0.366481,-0.926500,...,-0.577350,-0.581794,-0.319423,-0.375274,-0.304423,-0.340118,-0.293405,1.811455,-0.485803,0
2,-0.255185,-0.724000,-0.077537,0.091233,0.294877,0.673881,2.262432,-0.623912,0.923374,-0.926500,...,1.732051,-0.581794,-0.319423,-0.375274,-0.304423,2.940160,-0.293405,-0.552042,-0.485803,0
3,0.583322,1.453559,-0.560394,0.091233,-0.675976,0.673881,2.262432,1.602791,0.923374,2.847493,...,1.732051,-0.581794,-0.319423,-0.375274,-0.304423,-0.340118,-0.293405,-0.552042,2.058448,1
4,0.223962,-0.724000,-0.318966,1.508178,1.265730,-0.048604,-0.076958,-0.623912,-1.025752,0.690925,...,1.732051,-0.581794,-0.319423,-0.375274,-0.304423,-0.340118,-0.293405,-0.552042,2.058448,0


In [18]:
# Create DiCE objects

continuous_features = X_train_smote.columns.tolist()

d = dice_ml.Data(
    dataframe=dice_train_df,
    continuous_features=continuous_features,
    outcome_name="Churn"
)

m = dice_ml.Model(
    model=rf_model,
    backend="sklearn"
)

exp = Dice(d, m, method="random")

In [19]:
# Select only high churn risk customers

high_risk_customers = final_recommendation_df[
    final_recommendation_df["Churn_Risk"] == "High Churn Risk"
]

print("High risk customer count:", high_risk_customers.shape[0])

high_risk_customers.head()

High risk customer count: 96


,Customer_Index,Churn_Probability,Churn_Risk,Customer_Behaviour_Segment,Priority_Score,Priority_Label,Final_Manager_Action
11,11,0.785,High Churn Risk,New and Developing Customers,78.5,Immediate Action,Provide onboarding support and first-purchase ...
41,41,0.735,High Churn Risk,Inactive Customers,83.5,Immediate Action,Use low-cost win-back campaign or automated re...
53,53,0.800,High Churn Risk,Previously Valuable Customers,100.0,Immediate Action,Run priority win-back campaign with personalis...
59,59,0.850,High Churn Risk,New and Developing Customers,85.0,Immediate Action,Provide onboarding support and first-purchase ...
79,79,0.760,High Churn Risk,High-Value Occasional Customers,81.0,Immediate Action,Send premium personalised retention offer.


In [20]:
# Define only actionable features for DiCE

features_to_vary = [
    "Tenure",
    "DaySinceLastOrder",
    "Complain",
    "SatisfactionScore",
    "CouponUsed",
    "OrderCount",
    "CashbackAmount"
]

In [21]:
# Test DiCE on one high-risk customer first

customer_index = int(high_risk_customers.iloc[0]["Customer_Index"])

query_instance = X_test_scaled.iloc[[customer_index]]

dice_exp = exp.generate_counterfactuals(
    query_instance,
    total_CFs=1,
    desired_class=0,
    features_to_vary=features_to_vary
)

dice_exp.visualize_as_dataframe()

100%|██████████| 1/1 [00:00<00:00,  2.10it/s]

Query instance (original outcome : 1)


,Tenure,CityTier,WarehouseToHome,HourSpendOnApp,NumberOfDeviceRegistered,SatisfactionScore,NumberOfAddress,Complain,OrderAmountHikeFromlastYear,CouponUsed,...,M_Score_3,M_Score_4,CustomerSegment_Frequent Low-Value Customers,CustomerSegment_High-Value Active Customers,CustomerSegment_High-Value Customers Losing Interest,CustomerSegment_High-Value Occasional Customers,CustomerSegment_Inactive Customers,CustomerSegment_New and Developing Customers,CustomerSegment_Previously Valuable Customers,Churn
0,-1.213479,-0.724,1.733179,0.091233,0.294877,1.396365,-0.856755,-0.623912,-1.025752,-0.387359,...,1.732051,-0.581794,-0.319423,-0.375274,3.284901,-0.340118,-0.293405,-0.552042,-0.485803,1



Diverse Counterfactual set (new outcome: 0)


,Tenure,CityTier,WarehouseToHome,HourSpendOnApp,NumberOfDeviceRegistered,SatisfactionScore,NumberOfAddress,Complain,OrderAmountHikeFromlastYear,CouponUsed,...,M_Score_3,M_Score_4,CustomerSegment_Frequent Low-Value Customers,CustomerSegment_High-Value Active Customers,CustomerSegment_High-Value Customers Losing Interest,CustomerSegment_High-Value Occasional Customers,CustomerSegment_Inactive Customers,CustomerSegment_New and Developing Customers,CustomerSegment_Previously Valuable Customers,Churn
0,1.547379,-0.724,1.733179,0.091233,0.294877,1.396365,-0.856755,-0.623912,-1.025752,-0.387359,...,1.732051,-0.581794,-0.319423,-0.375274,3.284901,-0.340118,-0.293405,-0.552042,-0.485803,0


In [22]:
# Generate DiCE counterfactuals for all high-risk customers

dice_results = []

for idx in high_risk_customers["Customer_Index"]:

    idx = int(idx)
    query_instance = X_test_scaled.iloc[[idx]]

    original_prob = rf_model.predict_proba(query_instance)[0][1]

    try:
        dice_exp = exp.generate_counterfactuals(
            query_instance,
            total_CFs=1,
            desired_class=0,
            features_to_vary=features_to_vary
        )

        cf_df = dice_exp.cf_examples_list[0].final_cfs_df

        if cf_df is not None and len(cf_df) > 0:

            cf_instance = cf_df.drop(columns=["Churn"], errors="ignore").iloc[0]
            cf_instance = cf_instance[X_test_scaled.columns]

            target_prob = rf_model.predict_proba(
                pd.DataFrame([cf_instance], columns=X_test_scaled.columns)
            )[0][1]

            changed = False

            for feature in features_to_vary:

                current_value = query_instance.iloc[0][feature]
                suggested_value = cf_instance[feature]

                if abs(current_value - suggested_value) > 0.01:

                    changed = True

                    dice_results.append({
                        "Customer_Index": idx,
                        "Original_Probability": round(original_prob, 3),
                        "Target_Probability": round(target_prob, 3),
                        "Feature_To_Change": feature,
                        "Current_Value_Scaled": round(current_value, 3),
                        "Suggested_Value_Scaled": round(suggested_value, 3),
                        "Status": "Success"
                    })

            if not changed:
                dice_results.append({
                    "Customer_Index": idx,
                    "Original_Probability": round(original_prob, 3),
                    "Target_Probability": round(target_prob, 3),
                    "Feature_To_Change": "No major feature change detected",
                    "Current_Value_Scaled": None,
                    "Suggested_Value_Scaled": None,
                    "Status": "Success"
                })

        else:
            dice_results.append({
                "Customer_Index": idx,
                "Original_Probability": round(original_prob, 3),
                "Target_Probability": None,
                "Feature_To_Change": None,
                "Current_Value_Scaled": None,
                "Suggested_Value_Scaled": None,
                "Status": "No counterfactual found"
            })

    except Exception as e:
        dice_results.append({
            "Customer_Index": idx,
            "Original_Probability": round(original_prob, 3),
            "Target_Probability": None,
            "Feature_To_Change": None,
            "Current_Value_Scaled": None,
            "Suggested_Value_Scaled": None,
            "Status": f"Failed: {str(e)}"
        })

dice_counterfactual_df = pd.DataFrame(dice_results)

dice_counterfactual_df.head()

100%|██████████| 1/1 [00:00<00:00,  2.69it/s]


,Customer_Index,Original_Probability,Target_Probability,Feature_To_Change,Current_Value_Scaled,Suggested_Value_Scaled,Status
0,11,0.785,0.135,Tenure,-1.213,3.201,Success
1,11,0.785,0.135,SatisfactionScore,1.396,0.094,Success
2,41,0.735,0.175,Tenure,-1.094,2.920,Success
3,41,0.735,0.175,CouponUsed,-0.387,3.664,Success
4,53,0.800,0.225,Tenure,-1.213,5.827,Success


In [26]:
print(dice_counterfactual_df["Feature_To_Change"].unique())
print("\nNumber of unique features:",
      dice_counterfactual_df["Feature_To_Change"].nunique())

['Tenure' 'SatisfactionScore' 'CouponUsed' 'CashbackAmount'
 'DaySinceLastOrder' 'Complain' 'OrderCount']

Number of unique features: 7


In [27]:
print(dice_counterfactual_df["Feature_To_Change"].value_counts())

Feature_To_Change
Tenure               88
DaySinceLastOrder    18
CashbackAmount       17
SatisfactionScore    12
CouponUsed           11
OrderCount            8
Complain              5
Name: count, dtype: int64


In [ ]:
# Add manager-readable counterfactual action

def explain_counterfactual_action(feature):

    if feature == "DaySinceLastOrder":
        return "Encourage the customer to place a new order sooner through a re-engagement campaign."

    elif feature == "Complain":
        return "Resolve the complaint and follow up with the customer."

    elif feature == "SatisfactionScore":
        return "Improve satisfaction through service recovery or personalised support."

    elif feature == "CouponUsed":
        return "Provide a targeted coupon or promotional incentive."

    elif feature == "OrderCount":
        return "Increase repeat purchases through loyalty rewards or reminder campaigns."

    elif feature == "CashbackAmount":
        return "Offer cashback or loyalty reward to encourage retention."

    elif feature == "No major feature change detected":
        return "Review customer manually; DiCE did not identify a major actionable feature change."

    else:
        return "Review customer behaviour and apply suitable retention action."


dice_counterfactual_df["Counterfactual_Manager_Action"] = dice_counterfactual_df[
    "Feature_To_Change"
].apply(explain_counterfactual_action)

dice_counterfactual_df.head()

In [ ]:
# Save Part 2 output

dice_counterfactual_df.to_csv(
    "dice_counterfactual_highrisk.csv",
    index=False
)

In [ ]:
# Final checks

print("DiCE status counts:")
print(dice_counterfactual_df["Status"].value_counts())

dice_counterfactual_df[
    [
        "Customer_Index",
        "Original_Probability",
        "Target_Probability",
        "Feature_To_Change",
        "Counterfactual_Manager_Action",
        "Status"
    ]
].head(20)